## **Bank Nifty Predictability Under Regulatory Expiry-Structure Reform**

### **Notebook 01: Data Ingestion and Expiry Calendar Construction**

[![Python](https://img.shields.io/badge/Python-3.10%2B-blue)](https://www.python.org/) [![Pandas](https://img.shields.io/badge/Pandas-Latest-green)](https://pandas.pydata.org/) [![yfinance](https://img.shields.io/badge/yfinance-Latest-orange)](https://github.com/ranaroussi/yfinance) [![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

---

**Scope:** Download Bank Nifty daily OHLCV data via yfinance, programmatically construct the historical options monthly expiry calendar across regulatory regime shifts, and generate the holiday calendar for date alignment.

---


### **Pipeline Context**

This is a sequential 9-notebook experimental workflow:

1. **01_data_collection.ipynb**
2. **02_data_cleaning_eda.ipynb**
3. **03_feature_engineering.ipynb**
4. **04_volatility_regime_detection.ipynb**
5. **05_baseline_models.ipynb**
6. **06_proposed_model.ipynb**
7. **07_ablation_study.ipynb**
8. **08_evaluation_and_diagnostics.ipynb**
9. **09_figures_for_paper.ipynb**

---
**Current Location:** 1. **01_data_collection.ipynb** <- *You are here*

---
**Research Project**: Bank Nifty Predictability Under Expiry Reforms  
**Date**: June 27, 2026  
**Status**: Experimental Protocol Active

## 1. Introduction

### 1.1 Purpose of This Notebook
This notebook serves as the **foundational data ingestion and validation stage** of our study. In derivatives markets, price behavior is strongly structured by calendar-driven contract expirations. To analyze these patterns, we download raw Bank Nifty index daily data (`^NSEBANK`) and merge it against a programmatically compiled options expiry calendar. Because the monthly expiry day changed repeatedly during this window (Thursdays to Wednesdays to Tuesdays), building this calendar represents a key empirical step.

### 1.2 Research Question and Hypothesis
> *"Does explicitly encoding regulatory derivatives expiry-cycle structures as model features improve next-day Bank Nifty price forecasting compared to standard baselines using only generic technical indicators?"*

**Null Hypothesis (H₀):** Models using derivatives calendar and volatility-regime features show no significant difference in next-day forecasting compared to technical-only baselines.

**Alternative Hypothesis (H₁):** Incorporating structural calendar and volatility-regime features yields a statistically significant improvement in forecasting accuracy.

### 1.3 What This Notebook Does NOT Do
To prevent leakage and preserve experimental boundaries, this notebook does **not**:
- Compute technical indicators or features (deferred to Notebook 03).
- Fit the Hidden Markov Model (deferred to Notebook 04).
- Train baseline or proposed models (deferred to Notebooks 05 and 06).

## 2. Environment Setup

### 2.1 Imports and Library Diagnostics

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import yfinance as yf

# Setup project path
project_dir = Path("e:/BN_Structure_Reforms/banknifty-expiry-volatility-ml")
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

from src.data_loader import load_data_pipeline
print("[OK] Libraries and project modules imported successfully.")
print(f"[OK] pandas version: {pd.__version__}")
print(f"[OK] numpy version: {np.__version__}")
print(f"[OK] yfinance version: {yf.__version__}")

[OK] Libraries and project modules imported successfully.
[OK] pandas version: 2.3.3
[OK] numpy version: 2.2.6
[OK] yfinance version: 1.4.1


## 3. Data Ingestion Pipeline

### 3.1 Fetch Market Data and Construct Calendars
We run the automated load pipeline to pull index data, construct the monthly expiry calendar under shifting rules, and identify trading holidays.

In [2]:
# Download Bank Nifty data, generate expiry dates, and identify holidays
df_ohlcv, df_expiry, df_holidays = load_data_pipeline()

[*********************100%***********************]  1 of 1 completed

Saved raw Bank Nifty data to E:\BN_Structure_Reforms\banknifty-expiry-volatility-ml\data\raw\banknifty_ohlcv_2023_2026.csv
Generated and saved expiry calendar to E:\BN_Structure_Reforms\banknifty-expiry-volatility-ml\data\raw\nse_expiry_calendar.csv
Identified and saved 47 holidays to E:\BN_Structure_Reforms\banknifty-expiry-volatility-ml\data\raw\nse_holidays.csv


## 4. Pipeline Outputs Verification

### 4.1 Market Data Summary

In [3]:
print(f"Market Data Shape: {df_ohlcv.shape}")
print(f"Market Date Range: {df_ohlcv['Date'].min()} to {df_ohlcv['Date'].max()}")
df_ohlcv.head(10)

Market Data Shape: (755, 6)
Market Date Range: 2023-06-01 to 2026-06-25


,Date,Close,High,Low,Open,Volume
0,2023-06-01,43790.199219,44179.949219,43706.148438,44172.050781,182900
1,2023-06-02,43937.851562,44088.898438,43812.550781,43997.300781,123900
2,2023-06-05,44101.648438,44266.148438,44074.199219,44103.250000,124500
3,2023-06-06,44164.550781,44236.199219,44009.699219,44156.699219,123600
4,2023-06-07,44275.300781,44346.300781,44143.500000,44318.699219,119200
5,2023-06-08,43995.250000,44459.250000,43917.898438,44208.300781,160200
6,2023-06-09,43989.000000,44220.101562,43932.601562,44045.750000,132300
7,2023-06-12,43944.199219,44124.949219,43874.050781,44035.199219,91400
8,2023-06-13,44079.851562,44138.148438,43889.699219,43889.699219,160000
9,2023-06-14,43988.000000,44212.351562,43956.148438,44181.648438,154600


### 4.2 Expiry Calendar Validation

In [4]:
print(f"Total Expiry Cycles Compiled: {len(df_expiry)}")
print("Sample of Shifting Expiry Rules (Note Thursday to Wednesday to Tuesday shifts):")
df_expiry.head(15)

Total Expiry Cycles Compiled: 37
Sample of Shifting Expiry Rules (Note Thursday to Wednesday to Tuesday shifts):


,Year,Month,ScheduledExpiry,ActualExpiry,Rule,HolidayAdjusted
0,2023,6,2023-06-29,2023-06-28,Last Thursday,1
1,2023,7,2023-07-27,2023-07-27,Last Thursday,0
2,2023,8,2023-08-31,2023-08-31,Last Thursday,0
3,2023,9,2023-09-28,2023-09-28,Last Thursday,0
4,2023,10,2023-10-26,2023-10-26,Last Thursday,0
5,2023,11,2023-11-30,2023-11-30,Last Thursday,0
6,2023,12,2023-12-28,2023-12-28,Last Thursday,0
7,2024,1,2024-01-25,2024-01-25,Last Thursday,0
8,2024,2,2024-02-29,2024-02-29,Last Thursday,0
9,2024,3,2024-03-27,2024-03-27,Last Wednesday,0


### 4.3 Trading Holidays Diagnostics

In [5]:
print(f"Total Identified Trading Holidays: {len(df_holidays)}")
df_holidays.head(10)

Total Identified Trading Holidays: 47


,Date,DayOfWeek
0,2023-06-29,Thursday
1,2023-08-15,Tuesday
2,2023-09-19,Tuesday
3,2023-10-02,Monday
4,2023-10-24,Tuesday
5,2023-11-14,Tuesday
6,2023-11-27,Monday
7,2023-12-25,Monday
8,2024-01-22,Monday
9,2024-01-26,Friday
